# Run pipeline (02 → 05)

Runs notebooks **02–05** using `data/trials_hddm_ready.tsv` and model-comparison CSVs.
If model metrics or `hddm_va_vat_combined.nc` are missing under `models/transition_prob_duration_10000samples/`, syncs from `example_data/`.

Notebook **01** (HDDM refitting) is skipped by default; set env `RUN_HDDM_FIT=1` to include it.
Notebook **05** uses the bundled synthetic `hddm_va_vat_combined.nc` when no local fit is available.

In [2]:
import os
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NB = PROJECT_ROOT / "notebooks"
MODEL_DIR = PROJECT_ROOT / "models" / "transition_prob_duration_10000samples"
EXAMPLE_SCRIPT = PROJECT_ROOT / "example_data" / "generate_example_data.py"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WINNER_NC = "hddm_va_vat_combined.nc"
EXECUTOR = SCRIPTS_DIR / "execute_notebook.py"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from nc_io import stage_netcdf

needs_sync = (
    not any(MODEL_DIR.glob("*_dic_rmse_percentiles.csv"))
    or not (MODEL_DIR / WINNER_NC).is_file()
)
if needs_sync:
    print("Syncing example model assets from example_data/ …")
    subprocess.check_call(
        [sys.executable, str(EXAMPLE_SCRIPT), "--sync-models"],
        cwd=str(PROJECT_ROOT),
    )

winner_nc = MODEL_DIR / WINNER_NC
if winner_nc.is_file():
    staged_nc = stage_netcdf(winner_nc)
    os.environ["HDDM_WINNER_IDATA_PATH"] = staged_nc
    print(f"Staged InferenceData for netCDF4: {staged_nc}")

steps = []
if os.environ.get("RUN_HDDM_FIT", "").strip() == "1":
    steps.append("01_model_estimation.ipynb")
else:
    print("Skipping 01_model_estimation (set RUN_HDDM_FIT=1 to include).")

steps.extend([
    "02_model_comparison.ipynb",
    "03_parameter_recovery.ipynb",
    "04_condition_effect_analysis.ipynb",
])

if winner_nc.is_file():
    steps.append("05_figures.ipynb")
else:
    print(f"Skipping 05_figures (missing {WINNER_NC}).")

# Notebook 05 needs hddm.generate.gen_rts (PPC). Prefer the registered "hddm"
# kernel (override via HDDM_NOTEBOOK_KERNEL).
KERNEL_NAME = os.environ.get("HDDM_NOTEBOOK_KERNEL", "hddm").strip() or "hddm"
TIMEOUT_S = int(os.environ.get("HDDM_PIPELINE_TIMEOUT", "3600"))
print(f"Execute via isolated subprocess; kernel={KERNEL_NAME}; timeout={TIMEOUT_S}s")

# Nested ExecutePreprocessor inside a live Jupyter kernel often hits Windows ZMQ
# Empty → CellTimeoutError. Running scripts/execute_notebook.py in a subprocess
# avoids that. Agg + HDDM_PIPELINE_NONINTERACTIVE prevent plt.show() blocking.
env = os.environ.copy()
env.setdefault("MPLBACKEND", "Agg")
env["HDDM_PIPELINE_NONINTERACTIVE"] = "1"

for s in steps:
    print(f"\n=== {s} ===", flush=True)
    t0 = time.perf_counter()
    subprocess.check_call(
        [
            sys.executable,
            str(EXECUTOR),
            str(NB / s),
            "--kernel",
            KERNEL_NAME,
            "--timeout",
            str(TIMEOUT_S),
        ],
        cwd=str(NB),
        env=env,
    )
    print(f"    finished in {time.perf_counter() - t0:.1f}s", flush=True)

print("\nDone. See results/ and figures/.")


Staged InferenceData for netCDF4: C:\Users\lei\AppData\Local\Temp\hddm_analysis_idata\hddm_va_vat_combined.nc
Skipping 01_model_estimation (set RUN_HDDM_FIT=1 to include).
ExecutePreprocessor kernel: hddm

=== 02_model_comparison.ipynb ===

=== 03_parameter_recovery.ipynb ===

=== 04_condition_effect_analysis.ipynb ===

=== 05_figures.ipynb ===


CellTimeoutError: A cell timed out while it was being executed, after 3600 seconds.
The message was: Cell execution timed out.
Here is a preview of the cell contents:
-------------------
['# ==============================', '# Run: generate PPC figure + combined panel', '# ==============================', '', 'hddm_df_use = resolve_hddm_data()']
...
['    savepath=os.path.join(FIG_OUTDIR, "Figure_combined_v_a_DDM_2x3"),', ')', '', 'print("Saved figures to:", os.path.abspath(FIG_OUTDIR))', 'plt.show()']
-------------------
